# 1. Hiểu baseline CLIP tìm kiếm sản phẩm

**Mục tiêu:** giải thích được luồng **ảnh → xử lý ảnh → 512 số → so sánh → Top-5 → kiểm tra ID**.
*Query* là ảnh cần tìm; *gallery* là kho ảnh để tìm trong đó. Chỉ cần biết Python và mảng số cơ bản.

**Cách chạy:** mở riêng file này trong VS Code/Jupyter, chọn Python của
`visual-product-search-pilot/.venv/Scripts/python.exe`, rồi đọc/chạy **8 ô từ trên xuống**.
Không cần chạy `deepfashion_clip_baseline.ipynb` hay `scripts/pilot.py` trước: baseline đã hoàn tất.
Ô 2 đọc dữ liệu/cache; ô 4 tính embedding cho **một ảnh**; ô 5 tìm Top-5; ô 7 xem hai ca thật và Recall.
Ô 1, 3, 6, 8 chỉ cần đọc. File phụ `vps/learning.py` được import tự động, không chạy riêng.

**Các file dùng lại:** `../img/`, `../Eval/list_eval_partition.txt`, `.cache/clip/ViT-B-32.pt`,
`cache/embeddings/`, `reports/examples_manifest.json`, `reports/metrics.json`
(đường dẫn tính từ thư mục dự án). Nếu chuyển dữ liệu sang chỗ khác, sửa `DATA` ở ô 2.

**Vì sao không cần training?** CLIP đã được học trước từ các cặp ảnh–văn bản.
Ở đây chỉ dùng nhánh ảnh và giữ nguyên trọng số (*frozen* = đóng băng).
Không có optimizer, cập nhật trọng số hay nhãn dùng để dạy lại model; ID chỉ dùng để chấm kết quả.
Notebook chỉ đọc artifact đã có, không ghi đè baseline và không đo lại toàn bộ benchmark.


In [3]:
# 2. Dùng loader, embedding và hai ví dụ của thí nghiệm cũ
%matplotlib inline
from pathlib import Path
import json, sys
import numpy as np
import torch
from PIL import Image

ROOT = Path.cwd().resolve()
if not (ROOT / "vps").is_dir():
    ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT / "visual-product-search-pilot"
DATA = ROOT.parent  # Thư mục môn học đang chứa img/ và Eval/
sys.path.insert(0, str(ROOT))
from vps.annotations import parse_partition, validate_splits
from vps.embed import load_model
from vps.metrics import l2_normalize, retrieve, recall_at
from vps.learning import read_embeddings, show_preprocessing, show_top5

annotation = DATA / "Eval/list_eval_partition.txt"
records = parse_partition(annotation.read_text())
assert validate_splits(records)["query_items_without_gallery_positive"] == 0
queries = [r for r in records if r.split == "query"]
gallery = [r for r in records if r.split == "gallery"]
Q = read_embeddings(ROOT, "query", queries, annotation)
G = read_embeddings(ROOT, "gallery", gallery, annotation)
manifest = json.loads((ROOT / "reports/examples_manifest.json").read_text())
cases = [manifest["groups"]["success_top1"][0], manifest["groups"]["fail_top5"][0]]
query = next(r for r in queries if r.path == cases[0]["path"])
gallery_ids = [r.item_id for r in gallery]
print(f"Q: {Q.shape} = 14.218 query × 512 số; G: {G.shape} = 12.612 ảnh kho × 512 số")
print("Ảnh sẽ minh hoạ:", query.path, "| ID:", query.item_id)


ModuleNotFoundError: No module named 'vps'

## 3. Từ ảnh đến embedding: cần hiểu những kích thước nào?

*Tensor* là mảng số nhiều chiều; *shape* cho biết số phần tử theo từng chiều.

| Biến | Shape | Ý nghĩa |
|---|---|---|
| Ảnh gốc | 256 × 256 × 3 | Cao × rộng × 3 kênh màu RGB |
| `batch` | **1 × 3 × 224 × 224** | 1 ảnh, kênh màu đặt trước cao/rộng |
| `vector`, `q` | **1 × 512** | 1 ảnh được mô tả bằng 512 số |
| `G.T` | **512 × 12.612** | Đổi hàng/cột của kho embedding để nhân ma trận |
| `scores` | **1 × 12.612** | Mỗi ảnh gallery có một điểm so với query |

`preprocess` do **chính CLIP cung cấp**: resize bicubic, crop giữa ảnh thành 224×224,
đổi sang RGB/tensor rồi chuẩn hoá giá trị màu (*normalize*). Chỉ chia pixel cho 255 chưa đủ:
còn cần chuẩn hoá từng kênh màu theo thông số CLIP.
Hình bên dưới hoàn tác normalize **chỉ để xem ảnh**, không đổi đầu vào của model.

Embedding không phải 512 tên loại áo/quần: đó là biểu diễn số model đã học.
Sau khi đưa vector về độ dài 1 (*L2-normalize*), **cosine similarity = tích vô hướng**.
Vì vậy `q @ G.T` so sánh một ảnh với cả kho. Điểm càng lớn thì model thấy càng giống;
điểm 0,95 **không có nghĩa** là xác suất đúng 95%.


In [ ]:
# 4. Chỉ chạy CLIP cho MỘT ảnh; dùng đúng checkpoint và preprocessing của baseline
torch.set_num_threads(2)
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess, model_info = load_model(ROOT / ".cache/clip", device)
with Image.open(DATA / query.path) as source:
    image = source.convert("RGB")
batch = preprocess(image).unsqueeze(0)  # Thêm chiều số lượng ảnh: [3,224,224] → [1,3,224,224]
with torch.inference_mode():  # Chỉ suy luận, không tính gradient để training
    vector = model.encode_image(batch.to(device)).float().cpu().numpy()
q = l2_normalize(vector)  # Giữ hướng vector, đưa độ dài về 1
print("Thiết bị:", device, "| Trọng số đóng băng:", model_info["frozen"])
print("batch:", tuple(batch.shape), "| vector:", vector.shape, "| q:", q.shape)
print(f"Độ dài q sau chuẩn hoá: {np.linalg.norm(q[0]):.4f}")
show_preprocessing(image, batch, preprocess, q)


In [ ]:
# 5. So với TOÀN BỘ gallery, bằng hàm retrieval sẵn có
scores = q @ G.T  # [1,512] × [512,12612] → [1,12612]
top5 = retrieve(q, G, [query.item_id], gallery_ids, k=5)
print("Ma trận điểm cosine:", scores.shape, "| Vị trí Top-5:", top5["top_idx"].shape)
print("ID query:", query.item_id)
print("ID Top-5:", [gallery[int(j)].item_id for j in top5["top_idx"][0]])
show_top5(query, gallery, top5, DATA, title="Một ảnh vừa được trích xuất embedding → Top-5")
# Điểm có thể lệch rất nhỏ giữa GPU/runtime; ô 7 dùng cache của lượt đo gốc.


## 6. Đúng sản phẩm khác với trông giống: Recall@1 và Recall@5

Một kết quả chỉ **đúng khi product ID trùng query**. Cùng màu hay cùng loại quần chưa đủ.
Hai ảnh khác nhau của cùng sản phẩm được phép nằm ở query và gallery; chính ảnh query không được nằm trong gallery.

- **Recall@1:** số query có ảnh đúng ở vị trí đầu / tổng query hợp lệ.
- **Recall@5:** số query có ít nhất một ảnh đúng trong 5 vị trí đầu / tổng query hợp lệ.

Mỗi query chỉ đóng góp **0 hoặc 1**, dù Top-5 có nhiều ảnh cùng ID.
Baseline cũ dùng 14.218 query, không loại query nào. Hai ví dụ chọn để học không thể thay thế phép đo đó.

Ô 7 dùng lại cache và `retrieve`/`recall_at`, chỉ tìm lại **hai query** trong toàn bộ gallery,
không encode lại ảnh. Hai query cùng `id_00007524`, nhưng một ảnh chụp quần, một ảnh toàn thân.
Khung xanh lá = đúng ID, đỏ = sai; khung xanh dương bổ sung một ảnh đúng để hiểu ca thất bại.

**Tự kiểm tra trước khi chạy:** ca thật `id_00000353` trong manifest có ảnh đúng đầu tiên ở hạng 4.
Query này đóng góp bao nhiêu vào Recall@1 và Recall@5? Ô sau có đáp án từ hàm đánh giá.


In [ ]:
# 7. Hai ca THẬT: chỉ dùng embedding đã lưu, giữ nguyên kết quả benchmark
positions = [next(i for i, r in enumerate(queries) if r.path == case["path"]) for case in cases]
saved = retrieve(Q[positions], G, [case["item_id"] for case in cases], gallery_ids, k=5)
for row, case in enumerate(cases):
    assert int(saved["first_pos_rank"][row]) == case["first_pos_rank"]
    assert [gallery[int(j)].path for j in saved["top_idx"][row]] == [t["path"] for t in case["top5"]]
    title = "Ca đúng: ảnh tập trung vào quần" if row == 0 else "Ca sai: ảnh toàn thân, positive ở hạng 186"
    show_top5(queries[positions[row]], gallery, saved, DATA, row=row,
              title=title, positive=case.get("known_positive"))

two_cases = recall_at(saved["first_pos_rank"])
print(f"CHỈ 2 ca minh hoạ: R@1 = {two_cases['R@1']:.0%}, R@5 = {two_cases['R@5']:.0%}; không phải benchmark.")
full = json.loads((ROOT / "reports/metrics.json").read_text())["main_official_split"]
for k in (1, 5):
    print(f"Kết quả ĐÃ ĐO toàn bộ: R@{k} = {full[f'hits@{k}']}/{full['eligible_queries']} = {full[f'R@{k}']:.2%}")
rank_four = manifest["groups"]["fail_top1_recovered_top5"][0]["first_pos_rank"]
answer = recall_at(np.array([rank_four]))
print(f"Đáp án ca hạng {rank_four}: đóng góp {int(answer['R@1'])} cho R@1 và {int(answer['R@5'])} cho R@5.")


## 8. Có thể cải thiện thế nào? — giải thích, chưa chạy thêm thí nghiệm

**Crop sản phẩm:** bounding box là hình chữ nhật đánh dấu quần/áo cần tìm.
Cắt vùng này rồi mới đưa vào CLIP có thể giảm ảnh hưởng của áo khác, người mẫu và nền.
Hai ảnh cùng ID ở trên gợi ý vấn đề đó, **chưa chứng minh** crop chắc chắn cải thiện Recall.
File `../list_bbox_inshop.txt` đã có sẵn; baseline hiện tại vẫn dùng ảnh đầy đủ.

**Metric learning:** học để embedding **cùng product ID gần nhau, khác ID xa nhau**
(ví dụ dùng contrastive loss hoặc triplet loss). Có thể học thêm một lớp biến đổi embedding hoặc fine-tune encoder.
Khác baseline frozen, cách này **cần training** và cặp ảnh/nhãn sản phẩm.
Chỉ dùng train identities để học, tách validation để chọn cách làm; giữ official query/gallery để đánh giá cuối.

**Khi giải thích với giảng viên, nêu ba ý:** CLIP biến ảnh thành vector 512 chiều;
cosine xếp hạng ảnh trong gallery; product ID quyết định hit/miss để tính Recall.
Baseline đã đo **R@1 = 45,51%, R@5 = 67,25%** trên split đầy đủ; hai ca minh hoạ không nói lên tỷ lệ lỗi chung.
Ảnh ở đây là ảnh in-shop: chưa thể suy ra chất lượng với ảnh điện thoại; cũng chưa loại trừ được trùng lặp với dữ liệu pretrain của CLIP.

**Đọc tiếp đúng chỗ:** `vps/annotations.py` (partition), `vps/embed.py` (model/preprocess/cache),
`vps/metrics.py` (cosine/Recall), `reports/BAO_CAO_PILOT.md` (kết quả và hạn chế).
